# ReAct Agent with Pydantic AI

A ReAct (Reasoning and Acting) agent interleaves reasoning with tool usage. The LLM decides which tools to call, observes results, and reasons about next steps until it can provide a final answer.

```mermaid
flowchart LR
    User([User]) --> Agent["Agent"]
    Agent --> LLM{"LLM"}
    LLM -->|"Tool call"| Tool["Tool Execution"]
    Tool -->|"Result"| LLM
    LLM -->|"Final answer"| User
```

Pydantic AI handles the ReAct loop internally. You just define tools and the agent manages the reasoning-action cycle automatically.

In [ ]:
import nest_asyncio

nest_asyncio.apply()

## Setup

In [ ]:
from dotenv import load_dotenv
from pydantic_ai import Agent

load_dotenv()

## Vanilla implementation

We build an agent with a `run_python_code` tool. Pydantic AI handles the reasoning loop automatically.

In [ ]:
agent = Agent(
    "openai:gpt-5-mini",
    system_prompt="You are a helpful assistant that can run python code.",
)


# THIS IS DANGEROUS, DO NOT USE IN PRODUCTION
@agent.tool_plain
def run_python_code(code: str) -> str:
    """Run arbitrary Python code including imports, assignments, and statements.
    Do not use any external libraries. Save your results as a variable.

    Args:
        code: Python code to run
    """
    import sys
    from io import StringIO

    old_stdout = sys.stdout
    sys.stdout = captured_output = StringIO()

    namespace = {}

    try:
        exec(code, namespace)

        output = captured_output.getvalue()

        if not output.strip():
            user_vars = {
                k: v
                for k, v in namespace.items()
                if not k.startswith("__") and k not in ["StringIO", "sys"]
            }
            if user_vars:
                if len(user_vars) == 1:
                    output = str(list(user_vars.values())[0])
                else:
                    output = str(user_vars)

        return output.strip() if output.strip() else "Code executed successfully"

    except Exception as e:
        return f"Error: {str(e)}"
    finally:
        sys.stdout = old_stdout


# THIS IS DANGEROUS, DO NOT USE IN PRODUCTION

In [ ]:
response = agent.run_sync(
    "Generate 10 random numbers normally distributed with mean 0 and standard deviation 10"
)
print(response.output)

In [ ]:
response = agent.run_sync("Sample 10 values from a uniform distribution")
print(response.output)

## Agent with multiple tools

You can give the agent multiple tools. Here we add a weather tool alongside the code runner.

In [ ]:
import requests
from typing import Literal
from pydantic import BaseModel


class Feedback(BaseModel):
    feedback: str
    status: Literal["OK", "REQUIRES FIXING"]


evaluator_agent = Agent(
    "openai:gpt-5-mini",
    system_prompt=(
        "You're a helpful assistant. Your task is to check if a given response follows the company guidelines. "
        "The company guidelines are that responses should be written in the style of a haiku. "
        "You should reply with 'OK' or 'REQUIRES FIXING' and a short explanation."
    ),
    output_type=Feedback,
)

multi_tool_agent = Agent(
    "openai:gpt-5-mini",
    system_prompt=(
        "You're a helpful assistant. Use the tools provided when relevant. "
        "Then draft a response and check if it follows the company guidelines. "
        "Only respond to the user after you've validated and modified the response if needed."
    ),
)


@multi_tool_agent.tool_plain
def get_weather(latitude: float, longitude: float) -> str:
    """Get the weather of a given latitude and longitude"""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}"
        f"&current=temperature_2m,wind_speed_10m"
    )
    data = response.json()
    return str(data["current"]["temperature_2m"])


@multi_tool_agent.tool_plain
def check_guidelines(drafted_response: str) -> Feedback:
    """Check if a given response follows the company guidelines"""
    response = evaluator_agent.run_sync(drafted_response)
    return response.output


response = multi_tool_agent.run_sync("What is the temperature in Madrid?")
print(response.output)

# Exercise

Build an agent with multiple tools (e.g., weather, calculator, web search). Test it with queries that require using more than one tool.